# Deteccion de lavado de dinero en remesas (PaySim + IBM AML)

Sistema de dos etapas: un autoencoder secuencial con atencion que aprende el
comportamiento normal de remesas (Etapa A), y un clasificador que hace transfer
learning desde ese encoder para distinguir lavado de dinero de comportamiento
legitimo (Etapa B). Notebook autocontenido, reproducible en Colab con GPU T4
gratuita en menos de 30 minutos.

## 1. Configuracion y entorno

In [ ]:
import sys, os, time, json, random, math, warnings, glob, copy
NOTEBOOK_START = time.time()

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kagglehub"], check=False)

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

print(f"torch: {torch.__version__}")
print(f"CUDA disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("GPU: ninguna (CPU) -- el entrenamiento sera mas lento")


In [ ]:
# Configuracion global. Todos los hiperparametros y flags viven aqui:
# nada de constantes magicas repartidas por el notebook.
CFG = dict(
    SEQUENCE_SOURCE="ibm_aml",     # "ibm_aml" | "paysim" -- ver diagnostico en seccion 2
    REBUILD_FROM_RAW=False,        # True = pipeline completo desde CSV crudo de Kaggle
    PROCESSED_URL="TODO_RELEASE_ASSET_URL",  # TODO: URL de un release con el .npz ya procesado
    MAX_LEN=32,                    # valor inicial; se recalcula con el percentil 90 en la seccion 3
    MIN_LEN=5,
    N_SENDERS=60_000,              # subconjunto de remitentes; ver justificacion en seccion 3
    HIDDEN=64,
    LATENT=64,
    BATCH=256,
    EPOCHS_A=15,
    EPOCHS_B=12,                   # subido de 10: con FREEZE_EPOCHS mas corto, deja mas epocas de fine-tuning real
    FREEZE_EPOCHS=1,               # bajado de 3: menos tiempo "desperdiciado" en fase congelada
    LR_HEAD=1e-3,
    LR_ENCODER=3e-4,               # subido de 1e-4: permite al encoder alejarse mas rapido de una inicializacion de Etapa A que resulto poco informativa (ver seccion 6)
    FOCAL_GAMMA=2.0,
    FOCAL_ALPHA=0.25,
    ALERT_RATE=0.01,               # capacidad operativa del equipo de cumplimiento
    SEEDS=[0, 1, 2],
    DEVICE="cuda" if torch.cuda.is_available() else "cpu",
)
CFG


> **Decision:** usar IBM AML como fuente principal de secuencias por remitente
> (`SEQUENCE_SOURCE = "ibm_aml"`), con PaySim como validacion secundaria.
>
> **Justificacion:** se confirma con el diagnostico de la seccion 2 -- en PaySim
> `nameOrig` es casi unico por fila, lo que hace imposible construir secuencias
> temporales por remitente con mas de una o dos transacciones.
>
> **Alternativa descartada:** usar PaySim como fuente principal (como sugiere el
> enunciado al llamarlo "dataset principal"). Se descarta porque el objetivo del
> Componente 1 es explicitamente representar el comportamiento *a lo largo del
> tiempo* de un remitente, algo que PaySim no permite construir de forma fiable.


In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
set_seed(CFG["SEEDS"][0])
try:
    torch.use_deterministic_algorithms(True)
    print("Determinismo total activado.")
except Exception as e:
    print(f"No se pudo forzar determinismo total (se continua sin el): {e}")


class Timer:
    """Mide y reporta el tiempo de un bloque; los bloques pesados deben reportarse."""
    def __init__(self, label):
        self.label = label

    def __enter__(self):
        self.t0 = time.time()
        return self

    def __exit__(self, *exc):
        print(f"[TIEMPO] {self.label}: {time.time() - self.t0:.1f}s")
